# 🌿 Olive Yield Across Mediterranean Countries: Environmental and Socioeconomic Factors

## Notebook 02: Gold Dataset Engineering

## Overview

This notebook transforms the validated enriched Silver dataset produced in Notebook "01_data_collection_and_consolidation" into an analytics-ready Gold dataset.

The Country–Year structure is preserved while derived features are created to represent patterns not captured directly by the source variables. Feature engineering focuses on three analytical dimensions:

- **Yield dynamics:** changes in olive yield over time;
- **Climate dynamics:** temporal changes in atmospheric conditions;
- **Soil conditions:** measures derived from soil moisture and soil temperature across different depth layers.

Each feature is created for a defined analytical purpose, with attention to interpretability, missing values, temporal consistency. The objective is to prepare the data for downstream analysis without introducing unnecessary complexity.

The resulting Gold dataset is validated and exported as the primary input for exploratory data analysis, predictive modelling, and Power BI reporting.

## Imports and Display Settings

In [1]:
# File and path handling
from pathlib import Path

# Data manipulation
import pandas as pd

# Notebook-wide display settings
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.2f}".format)

## Load Enriched Silver Dataset

The validated enriched Silver dataset produced in Notebook 01 is loaded as the input to Gold-layer feature engineering.

In [2]:
df = pd.read_csv("../data/silver/integrated_silver_enriched.csv")

print(f"Dataset shape: {df.shape}")

Dataset shape: (274, 22)


## Dataset Validation

The analytical base consists of annual country-level observations covering eleven Mediterranean olive-producing countries between 2000 and 2024.

In [3]:
validation_summary = pd.DataFrame({
    "Metric": [
        "Observations",
        "Features",
        "Countries",
        "Period Start",
        "Period End",
        "Duplicate Rows",
        "Columns with Missing Values",
        "Total Missing Values"
    ],
    "Value": [
        df.shape[0],
        df.shape[1],
        df["Country"].nunique(),
        df["Year"].min(),
        df["Year"].max(),
        df.duplicated().sum(),
        df.isna().any().sum(),
        df.isna().sum().sum()
    ]
})

validation_summary

,Metric,Value
0,Observations,274
1,Features,22
2,Countries,11
3,Period Start,2000
4,Period End,2024
5,Duplicate Rows,0
6,Columns with Missing Values,2
7,Total Missing Values,6


## Missing-Value Investigation

This section examines the missing values in the dataset before feature engineering.

In [4]:
missing_values = (
    df.isna()
      .sum()
      .sort_values(ascending=False)
      .to_frame("Missing Values")
)

missing_values[missing_values["Missing Values"] > 0]

,Missing Values
Agriculture_value_added_pct_GDP,4
GDP_per_capita,2


Six missing values are present in the enriched Silver dataset:

- Agriculture_value_added_pct_GDP: 4;
- GDP_per_capita: 2.

No imputation or record removal is performed in this notebook. These values are retained for explicit treatment, if required, during downstream analysis or modelling.

## Feature Engineering Roadmap

To support subsequent analysis and predictive modeling, the Gold dataset is enriched through three complementary feature groups:

- Yield Dynamics
- Soil Conditions
- Climate Dynamics

The engineered features are designed to remain interpretable and suitable for downstream analysis.

### Yield Dynamics Features

To capture temporal changes in olive yield, two yield-dynamics features are created:

- Previous-Year Yield: records the olive yield from the preceding observation for each country.
- Yield Growth Rate: measures the relative year-to-year change in yield.

These features provide historical performance information for analysing yield persistence, growth, and decline over time.

In [5]:
# Previous-Year Yield

df = df.sort_values(["Country", "Year"])

df["Previous_Year_Yield"] = (df.groupby("Country")["Yield_kg_per_ha"].shift(1))

# Validation
df[
    [
        "Country",
        "Year",
        "Yield_kg_per_ha",
        "Previous_Year_Yield"
    ]
].head()


,Country,Year,Yield_kg_per_ha,Previous_Year_Yield
0,Algeria,2000,1291.70,NaN
1,Algeria,2001,1130.50,1291.70
2,Algeria,2002,1007.20,1130.50
3,Algeria,2003,799.30,1007.20
4,Algeria,2004,2071.20,799.30


The first observation of each country series contains a missing value because no earlier yield observation is available.

In [6]:
# Yield Growth Rate

df["Yield_Growth_Rate"] = (
    df.groupby("Country")["Yield_kg_per_ha"]
      .pct_change()
)

# Validation

df[
    [
        "Country",
        "Year",
        "Yield_kg_per_ha",
        "Yield_Growth_Rate"
    ]
].head()

,Country,Year,Yield_kg_per_ha,Yield_Growth_Rate
0,Algeria,2000,1291.70,NaN
1,Algeria,2001,1130.50,-0.12
2,Algeria,2002,1007.20,-0.11
3,Algeria,2003,799.30,-0.21
4,Algeria,2004,2071.20,1.59


Positive values in Yield Growth Rate indicate an increase in yield, while negative values indicate a decrease.

## Soil Condition Features

Soil conditions play an important role in olive cultivation by influencing water availability, root development, and overall tree performance.

The dataset includes soil moisture and soil temperature measurements at multiple depths. To provide a broader representation of root-zone conditions, weighed soil indicators are created while preserving the original measurements.

The following features are introduced:
  
- Weighted Soil Moisture (0–100 cm): This value is calculated by weighting each layer's moisture content by its thickness (7 cm, 21 cm, and 72 cm), summing them up, and dividing by the total depth of 100 cm.  
- Weighted Soil Temperature (7–100 cm): This value is calculated by focusing on the sub-surface layers, multiplying each temperature by its respective thickness (21 cm and 72 cm), and dividing the sum by the combined depth of 93 cm.

These variables complement the existing depth-specific measurements rather than replace them.

In [7]:
soil_moisture_columns = [
    "Mean_Soil_Moisture_0_7cm",
    "Mean_Soil_Moisture_7_28cm",
    "Mean_Soil_Moisture_28_100cm"
]

df["Weighted_Average_Soil_Moisture_0_100cm"] = (
    7  * df["Mean_Soil_Moisture_0_7cm"]
    + 21 * df["Mean_Soil_Moisture_7_28cm"]
    + 72 * df["Mean_Soil_Moisture_28_100cm"]
) / 100

# Validation
df.head()

,Country,Year,Area_Harvested_ha,Production_tonnes,Yield_kg_per_ha,Mean_Temperature_C,Mean_Dewpoint_C,Mean_Wind_Speed_m_s,Total_Precipitation_mm,Total_Solar_Radiation_MJ_m2,Mean_Soil_Moisture_0_7cm,Mean_Soil_Moisture_7_28cm,Mean_Soil_Moisture_28_100cm,Mean_Soil_Temperature_7_28cm_C,Mean_Soil_Temperature_28_100cm_C,Agricultural_land_pct,Agriculture_value_added_pct_GDP,Forest_area_pct,GDP_per_capita,Population,Rural_population_pct,Temperature_change,Previous_Year_Yield,Yield_Growth_Rate,Weighted_Average_Soil_Moisture_0_100cm
0,Algeria,2000,168080.00,217112.00,1291.70,17.07,8.17,2.53,311.59,5548.20,0.19,0.21,0.20,16.96,16.97,16.80,8.40,0.66,1772.93,30903893.00,40.15,0.77,NaN,NaN,0.20
1,Algeria,2001,177220.00,200339.00,1130.50,17.35,8.85,2.53,400.80,5427.26,0.20,0.22,0.20,17.32,17.34,16.84,8.89,0.68,1896.30,31331221.00,39.38,1.79,1291.70,-0.12,0.20
2,Algeria,2002,190550.00,191926.00,1007.20,17.14,8.82,2.61,422.70,5404.46,0.20,0.21,0.18,16.97,16.92,16.73,8.41,0.69,1937.46,31750835.00,38.62,1.19,1130.50,-0.11,0.19
3,Algeria,2003,209730.00,167627.00,799.30,17.29,9.75,2.60,679.29,5150.41,0.23,0.25,0.22,17.15,17.20,16.75,8.46,0.71,2283.77,32175818.00,37.86,1.52,1007.20,-0.21,0.23
4,Algeria,2004,226337.00,468800.00,2071.20,16.63,9.41,2.54,596.05,5214.95,0.23,0.24,0.22,16.40,16.45,17.28,7.72,0.72,2816.99,32628286.00,37.10,0.92,799.30,1.59,0.23


Average_Soil_Moisture is the unweighted mean of the three available soil-moisture layers. The original depth-specific variables are retained.

In [8]:
soil_temperature_columns = [
    "Mean_Soil_Temperature_7_28cm_C",
    "Mean_Soil_Temperature_28_100cm_C"
]

df["Weighted_Average_Soil_Temperature_7_100cm"] = (
    21 * df["Mean_Soil_Temperature_7_28cm_C"]
    + 72 * df["Mean_Soil_Temperature_28_100cm_C"]
) / 93

# Validation
df.sample(10, random_state=42)

,Country,Year,Area_Harvested_ha,Production_tonnes,Yield_kg_per_ha,Mean_Temperature_C,Mean_Dewpoint_C,Mean_Wind_Speed_m_s,Total_Precipitation_mm,Total_Solar_Radiation_MJ_m2,Mean_Soil_Moisture_0_7cm,Mean_Soil_Moisture_7_28cm,Mean_Soil_Moisture_28_100cm,Mean_Soil_Temperature_7_28cm_C,Mean_Soil_Temperature_28_100cm_C,Agricultural_land_pct,Agriculture_value_added_pct_GDP,Forest_area_pct,GDP_per_capita,Population,Rural_population_pct,Temperature_change,Previous_Year_Yield,Yield_Growth_Rate,Weighted_Average_Soil_Moisture_0_100cm,Weighted_Average_Soil_Temperature_7_100cm
30,Egypt,2005,49000.00,310000.00,6326.50,20.53,11.56,3.86,30.49,5070.78,0.05,0.14,0.15,22.26,22.29,3.54,13.98,0.06,1105.54,81101004.00,57.04,0.51,6421.20,-0.01,0.14,22.29
164,Portugal,2015,351340.00,722893.00,2057.50,15.76,9.02,2.90,544.06,5381.34,0.23,0.24,0.25,16.35,16.23,39.72,1.98,36.15,19215.78,10358076.00,38.86,1.38,1292.40,0.59,0.25,16.26
194,Spain,2020,2623720.00,8137810.00,3101.60,15.60,7.51,2.69,548.40,5220.76,0.24,0.24,0.21,16.22,16.24,52.33,2.78,37.18,27233.94,47359424.00,20.30,2.02,2292.60,0.35,0.22,16.24
125,Morocco,2001,550000.00,420000.00,763.60,17.52,9.18,2.55,327.09,5695.31,0.18,0.23,0.23,17.74,17.68,68.05,11.88,12.38,1506.25,28814643.00,46.10,1.69,740.70,0.03,0.22,17.69
265,Turkiye,2016,845542.00,1730000.00,2046.00,13.54,4.84,2.48,618.77,5241.91,0.22,0.20,0.18,14.19,14.29,49.80,6.27,28.10,10984.36,79277962.00,11.90,1.58,2031.20,0.01,0.18,14.27
232,Tunisia,2008,1719800.00,1183000.00,687.90,18.49,10.37,2.97,315.15,5144.32,0.17,0.19,0.18,18.39,18.39,63.60,7.85,4.40,4255.05,10542635.00,33.93,1.22,584.80,0.18,0.19,18.39
259,Turkiye,2010,784031.00,1415000.00,1804.80,14.37,6.67,2.39,714.82,5165.46,0.24,0.24,0.23,14.81,14.75,50.69,8.91,27.39,10698.97,73142150.00,30.65,2.40,1658.10,0.09,0.23,14.76
201,Syria,2002,501500.00,940941.00,1876.30,17.93,9.19,3.15,411.36,5403.48,0.17,0.21,0.19,18.62,18.69,74.87,24.86,2.42,1189.98,17468332.00,47.30,0.86,1016.40,0.85,0.19,18.68
255,Turkiye,2006,711843.00,1766749.00,2481.90,12.55,4.98,2.36,568.87,5158.62,0.24,0.23,0.21,13.15,13.13,52.61,8.06,26.91,7990.08,70045349.00,35.71,0.90,1812.70,0.37,0.22,13.13
216,Syria,2017,692417.00,849919.00,1227.50,18.79,8.24,3.02,301.01,5554.05,0.14,0.17,0.12,19.81,19.64,75.80,40.64,2.84,851.50,19224668.00,25.46,1.09,966.30,0.27,0.13,19.68


Average_Soil_Temperature is the unweighted mean of the two available soil-temperature layers. It provides a concise cross-layer indicator while the original depth-specific variables are retained.

## Climate Dynamics Features

Variables measuring the direction and magnitude of environmental change are introduced:

- Temperature Yearly Change
- Precipitation Yearly Change
- Soil Moisture Yearly Change

In [9]:
# Temperature Yearly Change

df["Temperature_Yearly_Change"] = (
    df.groupby("Country")["Mean_Temperature_C"]
      .diff()
)

df.head()

,Country,Year,Area_Harvested_ha,Production_tonnes,Yield_kg_per_ha,Mean_Temperature_C,Mean_Dewpoint_C,Mean_Wind_Speed_m_s,Total_Precipitation_mm,Total_Solar_Radiation_MJ_m2,Mean_Soil_Moisture_0_7cm,Mean_Soil_Moisture_7_28cm,Mean_Soil_Moisture_28_100cm,Mean_Soil_Temperature_7_28cm_C,Mean_Soil_Temperature_28_100cm_C,Agricultural_land_pct,Agriculture_value_added_pct_GDP,Forest_area_pct,GDP_per_capita,Population,Rural_population_pct,Temperature_change,Previous_Year_Yield,Yield_Growth_Rate,Weighted_Average_Soil_Moisture_0_100cm,Weighted_Average_Soil_Temperature_7_100cm,Temperature_Yearly_Change
0,Algeria,2000,168080.00,217112.00,1291.70,17.07,8.17,2.53,311.59,5548.20,0.19,0.21,0.20,16.96,16.97,16.80,8.40,0.66,1772.93,30903893.00,40.15,0.77,NaN,NaN,0.20,16.97,NaN
1,Algeria,2001,177220.00,200339.00,1130.50,17.35,8.85,2.53,400.80,5427.26,0.20,0.22,0.20,17.32,17.34,16.84,8.89,0.68,1896.30,31331221.00,39.38,1.79,1291.70,-0.12,0.20,17.33,0.28
2,Algeria,2002,190550.00,191926.00,1007.20,17.14,8.82,2.61,422.70,5404.46,0.20,0.21,0.18,16.97,16.92,16.73,8.41,0.69,1937.46,31750835.00,38.62,1.19,1130.50,-0.11,0.19,16.93,-0.21
3,Algeria,2003,209730.00,167627.00,799.30,17.29,9.75,2.60,679.29,5150.41,0.23,0.25,0.22,17.15,17.20,16.75,8.46,0.71,2283.77,32175818.00,37.86,1.52,1007.20,-0.21,0.23,17.19,0.15
4,Algeria,2004,226337.00,468800.00,2071.20,16.63,9.41,2.54,596.05,5214.95,0.23,0.24,0.22,16.40,16.45,17.28,7.72,0.72,2816.99,32628286.00,37.10,0.92,799.30,1.59,0.23,16.44,-0.66


Soil_Moisture_Yearly_Change measures the difference in average soil moisture compared with the preceding observation for each country.

Positive values indicate higher average soil moisture, while negative values indicate lower average soil moisture.

In [10]:
# Precipitation Yearly Change

df["Precipitation_Yearly_Change"] = (
    df.groupby("Country")["Total_Precipitation_mm"]
      .diff()
)

df.head()

,Country,Year,Area_Harvested_ha,Production_tonnes,Yield_kg_per_ha,Mean_Temperature_C,Mean_Dewpoint_C,Mean_Wind_Speed_m_s,Total_Precipitation_mm,Total_Solar_Radiation_MJ_m2,Mean_Soil_Moisture_0_7cm,Mean_Soil_Moisture_7_28cm,Mean_Soil_Moisture_28_100cm,Mean_Soil_Temperature_7_28cm_C,Mean_Soil_Temperature_28_100cm_C,Agricultural_land_pct,Agriculture_value_added_pct_GDP,Forest_area_pct,GDP_per_capita,Population,Rural_population_pct,Temperature_change,Previous_Year_Yield,Yield_Growth_Rate,Weighted_Average_Soil_Moisture_0_100cm,Weighted_Average_Soil_Temperature_7_100cm,Temperature_Yearly_Change,Precipitation_Yearly_Change
0,Algeria,2000,168080.00,217112.00,1291.70,17.07,8.17,2.53,311.59,5548.20,0.19,0.21,0.20,16.96,16.97,16.80,8.40,0.66,1772.93,30903893.00,40.15,0.77,NaN,NaN,0.20,16.97,NaN,NaN
1,Algeria,2001,177220.00,200339.00,1130.50,17.35,8.85,2.53,400.80,5427.26,0.20,0.22,0.20,17.32,17.34,16.84,8.89,0.68,1896.30,31331221.00,39.38,1.79,1291.70,-0.12,0.20,17.33,0.28,89.20
2,Algeria,2002,190550.00,191926.00,1007.20,17.14,8.82,2.61,422.70,5404.46,0.20,0.21,0.18,16.97,16.92,16.73,8.41,0.69,1937.46,31750835.00,38.62,1.19,1130.50,-0.11,0.19,16.93,-0.21,21.91
3,Algeria,2003,209730.00,167627.00,799.30,17.29,9.75,2.60,679.29,5150.41,0.23,0.25,0.22,17.15,17.20,16.75,8.46,0.71,2283.77,32175818.00,37.86,1.52,1007.20,-0.21,0.23,17.19,0.15,256.59
4,Algeria,2004,226337.00,468800.00,2071.20,16.63,9.41,2.54,596.05,5214.95,0.23,0.24,0.22,16.40,16.45,17.28,7.72,0.72,2816.99,32628286.00,37.10,0.92,799.30,1.59,0.23,16.44,-0.66,-83.24


Precipitation_Yearly_Change measures the difference in total annual precipitation compared with the preceding observation for each country.

Positive values indicate higher annual precipitation, while negative values indicate lower annual precipitation.

In [11]:
# Soil Moisture Yearly Change

df["Soil_Moisture_Yearly_Change"] = (
    df.groupby("Country")["Weighted_Average_Soil_Moisture_0_100cm"]
      .diff()
)

df.head()

,Country,Year,Area_Harvested_ha,Production_tonnes,Yield_kg_per_ha,Mean_Temperature_C,Mean_Dewpoint_C,Mean_Wind_Speed_m_s,Total_Precipitation_mm,Total_Solar_Radiation_MJ_m2,Mean_Soil_Moisture_0_7cm,Mean_Soil_Moisture_7_28cm,Mean_Soil_Moisture_28_100cm,Mean_Soil_Temperature_7_28cm_C,Mean_Soil_Temperature_28_100cm_C,Agricultural_land_pct,Agriculture_value_added_pct_GDP,Forest_area_pct,GDP_per_capita,Population,Rural_population_pct,Temperature_change,Previous_Year_Yield,Yield_Growth_Rate,Weighted_Average_Soil_Moisture_0_100cm,Weighted_Average_Soil_Temperature_7_100cm,Temperature_Yearly_Change,Precipitation_Yearly_Change,Soil_Moisture_Yearly_Change
0,Algeria,2000,168080.00,217112.00,1291.70,17.07,8.17,2.53,311.59,5548.20,0.19,0.21,0.20,16.96,16.97,16.80,8.40,0.66,1772.93,30903893.00,40.15,0.77,NaN,NaN,0.20,16.97,NaN,NaN,NaN
1,Algeria,2001,177220.00,200339.00,1130.50,17.35,8.85,2.53,400.80,5427.26,0.20,0.22,0.20,17.32,17.34,16.84,8.89,0.68,1896.30,31331221.00,39.38,1.79,1291.70,-0.12,0.20,17.33,0.28,89.20,-0.00
2,Algeria,2002,190550.00,191926.00,1007.20,17.14,8.82,2.61,422.70,5404.46,0.20,0.21,0.18,16.97,16.92,16.73,8.41,0.69,1937.46,31750835.00,38.62,1.19,1130.50,-0.11,0.19,16.93,-0.21,21.91,-0.02
3,Algeria,2003,209730.00,167627.00,799.30,17.29,9.75,2.60,679.29,5150.41,0.23,0.25,0.22,17.15,17.20,16.75,8.46,0.71,2283.77,32175818.00,37.86,1.52,1007.20,-0.21,0.23,17.19,0.15,256.59,0.04
4,Algeria,2004,226337.00,468800.00,2071.20,16.63,9.41,2.54,596.05,5214.95,0.23,0.24,0.22,16.40,16.45,17.28,7.72,0.72,2816.99,32628286.00,37.10,0.92,799.30,1.59,0.23,16.44,-0.66,-83.24,-0.00


Soil_Moisture_Yearly_Change measures the difference in average soil moisture compared with the preceding observation.

Positive values indicate higher average soil moisture, while negative values indicate lower average soil moisture.

### Climate Dynamic Features Validation

In [12]:
climate_dynamic_features = [
    "Temperature_Yearly_Change",
    "Precipitation_Yearly_Change",
    "Soil_Moisture_Yearly_Change"
]

df[climate_dynamic_features].isna().sum().to_frame("Missing Values")

,Missing Values
Temperature_Yearly_Change,11
Precipitation_Yearly_Change,11
Soil_Moisture_Yearly_Change,11


Yearly-change features generate missing values for the first observation of each country series because no previous-year reference exists.

These missing values are created by design and do not indicate data quality issues.

## Gold Dataset Assembly

The engineered features are combined with the original variables to produce the final Gold dataset.

In [ ]:
# Project directories
BASE_DIR = Path(".")
GOLD_DIR = BASE_DIR / "03_gold"

# Create gold directory if it does not exist
GOLD_DIR.mkdir(parents=True, exist_ok=True)

# Assemble Gold Dataset
gold_olive_dataset = df.copy()

print("Directories initialized successfully!")
print(f"Gold dataset shape: {gold_olive_dataset.shape}")

Directories initialized successfully!
Gold dataset shape: (274, 29)


## Gold Dataset Validation

In [14]:
engineered_features = [
    "Previous_Year_Yield",
    "Yield_Growth_Rate",
    "Weighted_Average_Soil_Moisture_0_100cm",
    "Weighted_Average_Soil_Temperature_7_100cm",
    "Temperature_Yearly_Change",
    "Precipitation_Yearly_Change",
    "Soil_Moisture_Yearly_Change"
]

gold_olive_dataset[engineered_features].head()

,Previous_Year_Yield,Yield_Growth_Rate,Weighted_Average_Soil_Moisture_0_100cm,Weighted_Average_Soil_Temperature_7_100cm,Temperature_Yearly_Change,Precipitation_Yearly_Change,Soil_Moisture_Yearly_Change
0,NaN,NaN,0.20,16.97,NaN,NaN,NaN
1,1291.70,-0.12,0.20,17.33,0.28,89.20,-0.00
2,1130.50,-0.11,0.19,16.93,-0.21,21.91,-0.02
3,1007.20,-0.21,0.23,17.19,0.15,256.59,0.04
4,799.30,1.59,0.23,16.44,-0.66,-83.24,-0.00


Missing values in the Gold dataset belong to two categories:

- six socioeconomic values inherited from the Silver dataset;
- expected values generated by lagged and yearly-change features where no preceding observation is available.

The engineered missing values are created by construction and do not indicate processing errors.

## Export Gold Dataset

The final Gold dataset is exported and will be used throughout the remaining stages of the project.

In [15]:
# Export Gold dataset

gold_olive_dataset.to_csv(
    GOLD_DIR / "gold_olive_dataset.csv",
    index=False,
    encoding="utf-8"
)

print("Gold dataset exported successfully.")

Gold dataset exported successfully.


## Executive Summary

This notebook transformed the validated enriched Silver dataset produced in Notebook 1 into an analytics-ready Gold dataset while preserving its Country–Year structure.

Seven features were created across three analytical dimensions:

- **yield dynamics:** Previous_Year_Yield and Yield_Growth_Rate;
- **soil conditions:** Average_Soil_Moisture and Average_Soil_Temperature;
- **climate dynamics:** Temperature_Yearly_Change, Precipitation_Yearly_Change, and Soil_Moisture_Yearly_Change.

The six socioeconomic missing values inherited from the Silver dataset were retained without imputation. Additional missing values in the lagged and yearly-change features are expected where no preceding observation is available for a country.

The final Gold dataset was validated and exported as gold_olive_dataset.csv for exploratory data analysis, predictive modelling, and Power BI reporting.